In [2]:
import os
import warnings

# Suppress tqdm warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Disable Hugging Face telemetry and download warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"



import sys
sys.path.append("../")

import pandas as pd
from app.services.embedding_service import compute_semantic_similarity
from app.services.scoring_engine import compute_match_score
from app.services.resume_parser import get_resume_text

df = pd.read_csv("../data/Resume.csv")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9573.70it/s]


In [3]:
similarity = compute_semantic_similarity(
    "Experienced Python backend developer with FastAPI",
    "Looking for a backend engineer skilled in Python"
)
print(f"Similarity: {similarity:.4f}")

Similarity: 0.5985


In [4]:
job_description = """
We are looking for a backend engineer with strong Python skills,
experience with FastAPI or Django, and familiarity with PostgreSQL
and Docker. 3+ years of experience preferred.
"""
required_skills = ["python", "fastapi", "postgresql", "docker"]
required_experience_years = 3

sample_df = df.sample(n=15, random_state=42).copy()

results = []
for _, row in sample_df.iterrows():
    text = get_resume_text(raw_text=row["Resume_str"])
    score_data = compute_match_score(
        resume_text=text,
        job_description=job_description,
        required_skills=required_skills,
        resume_experience_years=None,
        required_experience_years=required_experience_years,
    )
    score_data["ID"] = row["ID"]
    score_data["category"] = row["Category"]
    results.append(score_data)

results_df = pd.DataFrame(results)
results_df[["ID", "category", "match_score", "skill_overlap", "semantic_similarity"]].sort_values(
    "match_score", ascending=False
)

,ID,category,match_score,skill_overlap,semantic_similarity
13,73030450,ARTS,0.1627,0.0,0.4067
6,17199951,DESIGNER,0.1064,0.0,0.2661
5,39237915,BUSINESS-DEVELOPMENT,0.0951,0.0,0.2378
0,99244405,TEACHER,0.0935,0.0,0.2338
12,20552814,SALES,0.0925,0.0,0.2311
4,11065180,BANKING,0.0860,0.0,0.2150
14,14508237,AUTOMOBILE,0.0790,0.0,0.1975
3,19007667,CHEF,0.0751,0.0,0.1876
1,17562754,DIGITAL-MEDIA,0.0706,0.0,0.1765
2,30311725,CONSTRUCTION,0.0705,0.0,0.1762


In [5]:
df_sample2 = df.sample(n=50, random_state=1).copy()
scores = []
for _, row in df_sample2.iterrows():
    text = get_resume_text(raw_text=row["Resume_str"])
    result = compute_match_score(text, job_description, required_skills)
    scores.append(result["match_score"])

df_sample2["match_score"] = scores
df_sample2.groupby("Category")["match_score"].mean().sort_values(ascending=False)

Category
AVIATION                  0.368900
BPO                       0.343400
INFORMATION-TECHNOLOGY    0.293350
CONSULTANT                0.286700
BUSINESS-DEVELOPMENT      0.281263
ENGINEERING               0.278800
HR                        0.278300
AGRICULTURE               0.278100
CONSTRUCTION              0.276150
DIGITAL-MEDIA             0.273250
HEALTHCARE                0.272475
BANKING                   0.271200
FINANCE                   0.270867
APPAREL                   0.269000
ARTS                      0.266520
ADVOCATE                  0.264650
PUBLIC-RELATIONS          0.256267
CHEF                      0.254800
ACCOUNTANT                0.251600
FITNESS                   0.240833
TEACHER                   0.235600
AUTOMOBILE                0.215200
Name: match_score, dtype: float64

In [6]:
from app.services import scoring_engine

scoring_engine.WEIGHT_SKILL_OVERLAP = 0.5
scoring_engine.WEIGHT_SEMANTIC_SIMILARITY = 0.3
scoring_engine.WEIGHT_EXPERIENCE_MATCH = 0.2

# Re-run Cell 4's loop after changing weights to compare